# NLP Exercises (Part 2)

We have 2 exercises in this section. The exercises are:

4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before 

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [36]:
import numpy as np
import spacy

class BagOfWords:
    __nlp = spacy.load("en_core_web_sm")

    def __init__(self):
        self.__feature_names = []

    def fit_transform(self, corpus: list):
        tokenized = []
        vocabulary = set()

        for text in corpus:
            doc = self.__nlp(text)
            tokens = [token.text.lower() for token in doc if token.is_alpha]
            tokenized.append(tokens)
            vocabulary.update(tokens)

        self.__feature_names = sorted(vocabulary)
        index = {word: i for i, word in enumerate(self.__feature_names)}

        matrix = np.zeros((len(corpus), len(self.__feature_names)), dtype=int)
        for row, tokens in enumerate(tokenized):
            for token in tokens:
                matrix[row, index[token]] += 1

        return matrix

    def get_feature_names(self) -> list:
        return self.__feature_names

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]

vectorizer = BagOfWords()

X = vectorizer.fit_transform(corpus)
print(X)

print(vectorizer.get_feature_names())
print(len(vectorizer.get_feature_names()))

[[0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [1 0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]
['a', 'as', 'bag', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'really', 'see', 'sparse', 'the', 'third', 'this', 'throughout', 'us', 'words', 'you']
31


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [37]:
import nltk
nltk.download('book')
from nltk.book import *
import random
import re

N = 5
SEP = " "

wall_street = text7.tokens
tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    return list(filter(compiled_pattern.match, tokens))

tokens = cleanup()

def build_ngrams():
    ngrams = []
    for i in range(len(tokens) - N + 1):
        ngrams.append(tokens[i:i + N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}
    for ngram in ngrams:
        token_seq = SEP.join(ngram[:-1])
        last_token = ngram[-1]
        if token_seq not in counts:
            counts[token_seq] = {}
        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0
        counts[token_seq][last_token] += 1
    return counts

def next_word(text, N, counts):
    token_seq = SEP.join(text.split()[-(N - 1):])
    if token_seq not in counts:
        token_seq = random.choice(list(counts.keys()))
    choices = list(counts[token_seq].items())
    total = sum(weight for _, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight
        if upto > r:
            return choice
    return choices[-1][0]

[nltk_data] Downloading collection 'book'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/kkula/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package brown to
[nltk_data]    |     /Users/kkula/nltk_data...
[nltk_data]    |   Package brown is already up-to-date!
[nltk_data]    | Downloading package chat80 to
[nltk_data]    |     /Users/kkula/nltk_data...
[nltk_data]    |   Package chat80 is already up-to-date!
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /Users/kkula/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package conll2000 to
[nltk_data]    |     /Users/kkula/nltk_data...
[nltk_data]    |   Package conll2000 is already up-to-date!
[nltk_data]    | Downloading package conll2002 to
[nltk_data]    |     /Users/kkula/nltk_data...
[nltk_data]    |   Package conll2002 is already up-to-date!
[nltk_data]    | Downloading package dependency_t

In [38]:
def clean_generated(text):
    text = re.sub(r"\s+([.!?])", r"\1", text)
    pieces = [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]
    pieces = [p[0].upper() + p[1:] if p else p for p in pieces]
    return " ".join(pieces)

N = 5
SEP = " "
sentence_count = 5

ngrams = build_ngrams()
counts = ngram_freqs(ngrams)
start_seq = random.choice(list(counts.keys()))
generated = start_seq

sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith((".", "!", "?")) else 0

generated = clean_generated(generated)
print(generated)

The ability to regenerate damaged tissues or to turn off genes that cause cancer or to regulate genes that cause Down syndrome the leading cause of mental retardation according to an NIH summary. The NIH currently spends about 8 million annually on fetal-tissue research out of a total research budget of 8 billion. Rekindled hope that two New England states will allow broader interstate banking boosted Nasdaq bank stocks but the over-the-counter market was up only slightly in lackluster trading. The Nasdaq composite index added 1.01 to 456.64 on paltry volume of 118.6 million shares. In terms of volume it was an inauspicious beginning for November.
